# Aggregate impact function inputs to polygons

For main analysis / figures:
- QPLAD `tassumpoly` to impact regions (Carleton uses 0.25-deg GMFD)
- QPLAD `tassumpoly` to ADM1s (BHM uses 0.5-deg UDel)
- QPLAD `tasmaxbinF` to US counties (Behrer uses 4km PRISM)

For resolution dependency verification, see `2b_aggregate_1deg_to_shps.ipynb`.

In [8]:
import xarray as xr
import xagg as xa
import numpy as np
import pandas as pd
import geopandas as gpd
import os
import re
import glob
import zarr
import warnings
from tqdm.notebook import tqdm
from dask.diagnostics import ProgressBar

import distributed

from funcs_support import get_filepaths,get_params,utility_save
dir_list = get_params()

In [2]:
from distributed import Client

# Start dask client 
# (playing around with workers, threads for efficiency)
client = Client(n_workers=6,
    threads_per_worker=3,
    processes=True,
    memory_limit="8GB")
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/climate/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/climate/proxy/8787/status,Workers: 6
Total threads: 18,Total memory: 44.70 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:33355,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/climate/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:33931,Total threads: 3
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/climate/proxy/41211/status,Memory: 7.45 GiB
Nanny: tcp://127.0.0.1:46539,


In [3]:
# Get paths of BCD-ME files
bcd_fns = pd.DataFrame({'path':np.sort(glob.glob(dir_list['bcd_me']+'*.zarr'))})
bcd_fns['model'] = [re.split(r'\.',re.split(r'\_',fn)[-1])[0] for fn in bcd_fns.path]
bcd_fns['typ'] = [re.split(r'_',re.split(r'\/',fn)[-1])[2] for fn in bcd_fns.path.values]
modlist = bcd_fns['model'].unique()
bcd_fns = bcd_fns.set_index(['typ','model'])

In [4]:
params_proc = {'impact-regions':{'varnames':['tassumpoly'],'shp_fn':'impact-region'},
               'adm1':{'varnames':['tassumpoly'],'shp_fn':'country-fixed'},
               'adm2-us':{'varnames':['tasmaxbinF'],'shp_fn':'UScounties'}}


idxs = {'impact-regions':{'hierid':'locid'},
        'adm1':{'CNTRY_NAME':'locid'},
        'adm2-us':{'FIPS':'locid'}}

In [5]:
# Load population density
pop = xr.open_dataset(dir_list['aux']+'pop_data/glp00ag30.asc')
pop = xa.fix_ds(pop)
pop = pop.isel(band=0).band_data

# To get around an xagg error when weights are exactly 0
pop = pop.where(pop>0,0.0000000001) 

### Pre-processing of shapefiles
Primarily needed for BHM replication

In [6]:
if not os.path.exists(dir_list['aux']+'geo_data/country-fixed.shp'):
    
    # Load shapefile for insample regions
    gdf = gpd.read_file(dir_list['aux']+'geo_data/country.shp')
    
    rename_keys = {'Belarus':'Byelarus',
     'Brunei Darussalam':'Brunei',
     'Cabo Verde':'Cape Verde',
     'Channel Islands':['Jersey','Guernsey'],
     'Congo, Dem. Rep.':'Zaire',
     'Congo, Rep.':'Congo',
     "Cote d'Ivoire":'Ivory Coast',
     'Egypt, Arab Rep.':'Egypt',
     'Hong Kong SAR, China':'Hong Kong', #probably missing,
     'Iran, Islamic Rep.':'Iran',
     'Isle of Man':'Man, Isle of',
     'Korea, Rep.':'South Korea',
     'Kosovo':'Kosovo', # still part of Serbia in this sample
     'Kyrgyz Republic':'Kyrgyzstan',
     'Lao PDR':'Laos',
     'Macao SAR, China':'Macau',
     'Macedonia, FYR':'Macedonia',
     'Micronesia, Fed. Sts.':'Federated States of Micronesia',
     'Palau':'Pacific Islands (Palau)',
     'Russian Federation':'Russia',
     'Samoa':'Western Samoa',
     'Slovak Republic':'Slovakia',
     'Syrian Arab Republic':'Syria',
     'Tanzania':'Tanzania, United Republic of',
     'Timor-Leste':'Timor-Leste', # uh, still part of indonesia in this one... 
     'Venezuela, RB':'Venezuela',
     'West Bank and Gaza':['West Bank','Gaza Strip'],
     'Yemen, Rep.':'Yemen'}
    # Switch order
    rename_keys = {(tuple(v) if type(v) == list else v):k for k,v in rename_keys.items()}
    
    # Combine into single the ones that are multiple locs in the shapefile but not the repl dataset
    combine_locs = {k:v for k,v in rename_keys.items() if type(k) == tuple}
    for locs,joint_name in combine_locs.items():
        gdf_tmp = gdf.iloc[[c in locs for c in gdf.CNTRY_NAME.values],:]
        cols = [col for col in gdf_tmp.columns if col != 'geometry']
        
        gdf_joint = pd.concat([gdf_tmp.iloc[0].loc[[col for col in cols if gdf_tmp.dtypes[col] == object]],
                              gdf_tmp.loc[:,[col for col in cols if gdf_tmp.dtypes[col] in [float,int]]].sum()])
        gdf_joint = gpd.GeoDataFrame([gdf_joint],index=[gdf_tmp.index[0]],
                         geometry = [gdf_tmp.union_all()],
                                     crs = gdf_tmp.crs)
        gdf_joint.loc[:,'CNTRY_NAME'] = joint_name
        
        gdf = pd.concat([gdf.iloc[[c not in locs for c in gdf.CNTRY_NAME.values],:],
                   gdf_joint])
    gdf = gdf.sort_index()
    
    # Rename the rest of the locations as needed
    for idx in gdf.index:
        if gdf.loc[idx,'CNTRY_NAME'] in rename_keys:
            gdf.loc[idx,'CNTRY_NAME'] = rename_keys[gdf.loc[idx,'CNTRY_NAME']]
    
    # Save 
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore') # shp writing warning about POP_CNTRY stat
        gdf.to_file(dir_list['aux']+'geo_data/country-fixed.shp')

## Processing

In [10]:
# Load sample QPLAD file
ds_grid = xr.open_zarr(bcd_fns.loc['qdm-qplad'].iloc[0]['path'],decode_timedelta=False)[['lat','lon']]

# Create xagg weightmaps
for proc in params_proc:
    #------------------------
    # Setup -----------------
    #------------------------
    # Generate filenames
    output_fns = {mod:{var:{'nowght':f'{dir_list['proc']}aggs/{var}_agg_{mod}_ALLRUNS_ALLGWLs_{proc}.zarr',
                            'popwght':f'{dir_list['proc']}aggs/{var}_agg_{mod}_ALLRUNS_ALLGWLs_{proc}_popwght.zarr'}
                   for var in params_proc[proc]['varnames']}
             for mod in modlist}

    # Check for completed existence of all files for a given 
    # shapefile (utility_print adds an empty .done flag after saving)
    if np.all([np.all([np.all([os.path.exists(output_fns[k][v][t]+'/.done') 
                               for t in output_fns[k][v]]) 
                       for v in output_fns[k]]) 
               for k in output_fns.keys()]):
        print(f'All files generated for {proc}, skipped.')
        continue
    
    #------------------------
    # Generate weightmaps ---
    #------------------------
    # Load shapefile
    gdf = gpd.read_file(f'{dir_list['aux']}geo_data/{params_proc[proc]['shp_fn']}.shp')
    gdf['geometry'] = gdf.make_valid()

    # Keep relevant index columns
    gdf = gdf.loc[:,[*[k for k in idxs[proc]],'geometry']]

    # Rename index columns to standard names across analyses
    gdf = gdf.rename(idxs[proc],axis=1)

    # Create weightmap
    with warnings.catch_warnings():
        # Catches a weights bbox warning that doesn't affect this process
        warnings.filterwarnings('ignore')
        wms = {'nowght':xa.pixel_overlaps(ds_grid,gdf),
               'popwght':xa.pixel_overlaps(ds_grid,gdf,weights=pop)}

    #------------------------
    # Aggregate by model ----
    #------------------------
    for mod in modlist:
        for var in params_proc[proc]['varnames']:
            # Check for completed existence of files for a given
            # model / variable combination (utility_print adds an
            # empty .done flag after saving)
            path_exists = {wght:os.path.exists(output_fns[mod][var][wght]+'/.done') for wght in output_fns[mod][var]}
            if np.all([v for k,v in path_exists.items()]):
                print(f'All files for {mod} {var} aggregated to {proc} already generated, skipped.')
                continue
            proc_weights = [wght for wght,flag in path_exists.items() if not flag]
            
            # Load QDM-QPLAD downscaled data
            ds = xr.open_zarr(bcd_fns.loc[('qdm-qplad',mod),'path'],decode_timedelta=False)
            if var in ds:
                ds = ds[[var]]
            else:
                print(f'{var} not in {bcd_fns.loc[('qdm-qplad',mod),'path']}, skipped.')
                continue

        
            # Rechunk to allow geographic processing
            ds = ds.chunk({'lat':-1,'lon':-1,'proj_base':-1})

            # Aggregate
            with warnings.catch_warnings():
                # Stops inconsistent nan warning from xagg - which is just catching that
                # some gwls are all-nans for some idvs
                warnings.filterwarnings('ignore')
                
                with xa.set_options(impl='numba',silent=True):
                    aggs = {wght:xa.aggregate(ds,wms[wght])
                            for wght in proc_weights}
    
            # Move aggregated output to xarray dataset format
            aggs_out = {wght:agg.to_dataset()
                        for wght,agg in aggs.items()}
                    
            # Add experiment / run id back (secondary coordinates don't make it
            # through the xagg workflow) 
            for wght in aggs_out:
                aggs_out[wght] = aggs_out[wght].reset_index('idv').assign_coords({var:ds[var] for var in ['experiment','run']})
    
            #------------------------
            # Save ----
            #------------------------
            for wght in aggs_out:
                # Metadata
                if wght == 'nowght':
                    weightdesc = ''
                elif wght == 'popwght':
                    weightdesc = ' using pop. weights'
                aggs_out[wght].attrs = {'SOURCE':'aggregate_to_shps.ipynb',
                                        'DESCRIPTION':f'BCD-ME {mod} {var} data, aggregated{weightdesc} to {re.sub(r'\-', ' ',proc)} polygons'}
    
                # Chunking for zarr saving
                chunk_dims = ['idv','proj_base']
                full_dims = [dim for dim in aggs_out[wght][var].sizes if dim not in chunk_dims]
                aggs_out[wght] = aggs_out[wght].chunk({**{dim:1 for dim in chunk_dims},
                          **{dim:-1 for dim in full_dims}}
                        )
    
                # Output as zarr
                utility_save(aggs_out[wght].drop_encoding(),
                             output_fns[mod][var][wght],
                             zarr_mode = 'w',
                             save_kwargs = {'zarr_format':3})
                
        

All files generated for impact-regions, skipped.
All files generated for adm1, skipped.
creating polygons for each pixel...
lat/lon bounds not found in dataset; they will be created.
calculating overlaps between pixels and output polygons...
success!
creating polygons for each pixel...
regridding weights to data grid...
calculating overlaps between pixels and output polygons...
success!
All files for ACCESS-ESM1-5 tasmaxbinF aggregated to adm2-us already generated, skipped.
tasmaxbinF not in /glade/campaign/cgd/cas/schwarzwald/bcd_me/bcd_me_qdm-qplad_CESM2-WACCM.zarr, skipped.


/glade/u/home/schwarzwald/projects/impact_uncertainty/code/funcs_support.py:447: SerializationWarning: variable None has data in the form of a dask array with dtype=object, which means it is being loaded into memory to determine a data type that can be safely stored on disk. To avoid this, coerce this variable to a fixed-size dtype with astype() before saving it.
  ds.to_zarr(output_fn,**save_kwargs)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=8, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type 

/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tasmaxbinF_agg_CNRM-CM6-1_ALLRUNS_ALLGWLs_adm2-us.zarr saved!


/glade/u/home/schwarzwald/projects/impact_uncertainty/code/funcs_support.py:447: SerializationWarning: variable None has data in the form of a dask array with dtype=object, which means it is being loaded into memory to determine a data type that can be safely stored on disk. To avoid this, coerce this variable to a fixed-size dtype with astype() before saving it.
  ds.to_zarr(output_fn,**save_kwargs)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=8, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type 

/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tasmaxbinF_agg_CNRM-CM6-1_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr saved!


...

/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tasmaxbinF_agg_MPI-ESM1-2-LR_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr saved!


/glade/u/home/schwarzwald/projects/impact_uncertainty/code/funcs_support.py:447: SerializationWarning: variable None has data in the form of a dask array with dtype=object, which means it is being loaded into memory to determine a data type that can be safely stored on disk. To avoid this, coerce this variable to a fixed-size dtype with astype() before saving it.
  ds.to_zarr(output_fn,**save_kwargs)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=8, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type 

/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tasmaxbinF_agg_MRI-ESM2-0_ALLRUNS_ALLGWLs_adm2-us.zarr saved!


/glade/u/home/schwarzwald/projects/impact_uncertainty/code/funcs_support.py:447: SerializationWarning: variable None has data in the form of a dask array with dtype=object, which means it is being loaded into memory to determine a data type that can be safely stored on disk. To avoid this, coerce this variable to a fixed-size dtype with astype() before saving it.
  ds.to_zarr(output_fn,**save_kwargs)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=8, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type 

/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tasmaxbinF_agg_MRI-ESM2-0_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr saved!
